# Compare Multiple ASR Evaluation Runs

This notebook allows you to compare results from multiple evaluation runs with different configurations or parameters.

## Features:
- Load and compare multiple run results
- Visualize performance across runs
- Analyze parameter effects
- Generate comparison reports

---

## 1. Load Multiple Run Results

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import glob

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🔍 Looking for previous runs...")

# Find all run directories
results_dir = Path("results/comparisons")
run_dirs = [d for d in results_dir.iterdir() if d.is_dir() and d.name.startswith("run_")]
run_dirs.sort(reverse=True)  # Most recent first

print(f"Found {len(run_dirs)} previous runs:")
for i, run_dir in enumerate(run_dirs):
    print(f"  {i+1}. {run_dir.name}")

if len(run_dirs) == 0:
    print("❌ No previous runs found. Please run main evaluation notebook first.")

## 2. Select Runs to Compare

In [ ]:
# Select runs to compare (modify as needed)
# Options: 
# - Use all runs: selected_runs = run_dirs
# - Use specific runs: selected_runs = [run_dirs[0], run_dirs[1]]
# - Use N most recent: selected_runs = run_dirs[:3]

# Example: Compare 3 most recent runs
selected_runs = run_dirs[:3] if len(run_dirs) >= 3 else run_dirs

print(f"\n📊 Selected {len(selected_runs)} runs for comparison:")
for run_dir in selected_runs:
    print(f"  - {run_dir.name}")

# Load results from selected runs
all_results = []
run_configs = {}

for run_dir in selected_runs:
    # Load run metadata
    metadata_file = run_dir / f"run_metadata_{run_dir.name.replace('run_', '')}.json"
    
    if metadata_file.exists():
        with open(metadata_file, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        run_configs[run_dir.name] = metadata
        
        # Load results
        results_file = run_dir / f"asr_compare_{metadata['run_id']}.json"
        if results_file.exists():
            with open(results_file, 'r', encoding='utf-8') as f:
                run_results = json.load(f)
            
            # Add run information to each result
            for result in run_results:
                result['run_name'] = run_dir.name
                result['run_timestamp'] = metadata['timestamp']
                result['run_config'] = metadata.get('config', {})
            
            all_results.extend(run_results)
            print(f"  ✅ Loaded {len(run_results)} results from {run_dir.name}")
        else:
            print(f"  ❌ Results file not found for {run_dir.name}")
    else:
        print(f"  ❌ Metadata file not found for {run_dir.name}")

print(f"\n📈 Total results loaded: {len(all_results)}")